# PHASE 0.11 — Family-level uncertainty

## Phase 0.10 found something bigger than the verdict bug it was built to fix

The nested family holdout did not work either. **No legitimate selection rule ranks arms
consistently with the test outcome:**

| Selection rule | Spearman with test NLL |
|---|---|
| nested — 4 families fit, 2 held out | **+0.314** |
| strict — all 6 train families | **+0.029** |
| optimistic — the 6 test families | +1.000 (but it selects on what it reports) |

The held-out selection families produced NLL of 2.05–2.42 while the test families produced
0.76–1.03 — roughly 2.5× harder. **Corruption families are not exchangeable.** A rule tuned on
some families cannot rank calibrators for others, and no amount of nesting fixes that.

## And the results themselves are not stable

Between Phase 0.9 and Phase 0.10 the only change was fitting the calibrators on four of the six
train families instead of all six. Same data, same seed, same folds. C0 and C1 are byte-identical
across the two runs, confirming the difference is entirely the calibration family mix:

| Arm | NLL 0.9 → 0.10 | ECE 0.9 → 0.10 |
|---|---|---|
| C2 scalar / augmented | 0.7915 → 0.7793 | 0.0971 → **0.0382** (−61%) |
| C4 linear / raw *q* | 0.7352 → 0.7885 | 0.0770 → 0.0331 |
| C7q bounded / raw *q* | 0.7545 → 0.7630 | 0.0809 → 0.0329 |
| C5 linear / class one-hot | 0.8841 → 0.8629 | 0.0492 → 0.0432 |

Rank stability across the two runs: **+0.857 on NLL**, and **−0.595 on ECE** — the ECE ordering
essentially reversed. Which arm looks best on calibration error depends on which corruptions the
calibrator happened to be fitted on.

## What this means, and it is the most important finding in the whole audit

Every confidence interval reported so far resamples **images within a fixed set of six test
families**. Those intervals are ±0.005 on NLL and they are not wrong — they are answering a
narrow question: *would this hold on different images of these six corruptions?*

The paper asks a different question: *would this hold on six different corruptions?* For that,
the effective sample size is **n = 6**, not n = 5,400, and the family-to-family spread is large
enough that the answer may well be no.

**Reporting the image-level interval as if it supported a generalisation claim would have been
the single most serious error in the manuscript**, and it would have survived review — the
numbers look rigorous. Phase 0 caught it before any of it was written up. That is worth more than
any of the verdicts that came before.

## What this notebook does

Metrics are recorded **per test family**, and every comparison now carries two intervals: the
image-level bootstrap (conditional on this family set) and a family-level *t* interval over the
six families with 5 degrees of freedom. Comparisons that are significant on images but not across
families are listed explicitly, with their six per-family deltas printed.

The verdict has a new top branch. If **no legitimate selection rule reaches a rank correlation of
0.60** with the test outcome, it returns `UNDETERMINED` and names no method — because reading a
verdict off a misspecified selection is exactly how Phase 0.6, 0.7, 0.9 and 0.10 each produced a
wrong conclusion. A fifth repetition of that mistake is not worth risking.

## 0. Self-contained core module

In [ ]:
CORE_V2 = r"""
import math, hashlib
import numpy as np, cv2

def _clip(x): return np.clip(x, 0, 1)
def _mask3(mask, x): return mask[..., None] if x.ndim == 3 else mask

# ------------------------------------------------------------------ optics --
def defocus_blur(x, s):
    r = [1, 2, 3, 5, 7][s-1]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)).astype(np.float32)
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def motion_blur(x, s):
    ksz = [5, 9, 13, 19, 25][s-1]
    ang = np.random.uniform(0, 180)
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), ang, 1.0)
    k = cv2.warpAffine(k, M, (ksz, ksz))
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def vignetting(x, s):
    st = [0.15, 0.30, 0.45, 0.62, 0.80][s-1]
    h, w = x.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((xx-w/2)/(w/2))**2 + ((yy-h/2)/(h/2))**2)
    m = np.clip(1 - st*np.clip(r-0.4, 0, None)/0.6, 0, 1).astype(np.float32)
    return _clip(x * _mask3(m, x))

def lens_contamination(x, s):
    n = [3, 7, 13, 22, 34][s-1]; h, w = x.shape[:2]
    blurred = cv2.GaussianBlur(x, (0, 0), sigmaX=max(1.0, min(h, w)/40))
    mask = np.zeros((h, w), np.float32)
    for _ in range(n):
        c = (np.random.randint(0, w), np.random.randint(0, h))
        rad = np.random.randint(max(2, int(0.015*w)), max(4, int(0.07*w)))
        cv2.circle(mask, c, rad, 1.0, -1, lineType=cv2.LINE_AA)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(1.0, min(h, w)/60))
    m = _mask3(mask, x)                       # guard: 2-D input used to broadcast to (H,W,W)
    return _clip((x*(1-m) + blurred*m) * (1 - 0.25*m))

def vibration_jitter(x, s):
    amp = [0.6, 1.4, 2.6, 4.2, 6.5][s-1]; h, w = x.shape[:2]
    dx, dy = np.random.uniform(-amp, amp, 2)
    out = cv2.warpAffine(x, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    ksz = int(max(3, 2*round(amp)+1))
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M2 = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), math.degrees(math.atan2(dy, dx)), 1.0)
    k = cv2.warpAffine(k, M2, (ksz, ksz))
    return _clip(cv2.filter2D(out, -1, k/max(k.sum(), 1e-8)))

# ------------------------------------------------------------------ sensor --
def gaussian_noise(x, s):
    return _clip(x + np.random.normal(0, [0.03, 0.06, 0.10, 0.16, 0.24][s-1], x.shape))

def shot_noise(x, s):
    lam = [80, 35, 15, 7, 3][s-1]
    return _clip(np.random.poisson(x*lam)/float(lam))

def scanline_banding(x, s):
    amp = [0.03, 0.06, 0.11, 0.17, 0.25][s-1]; h = x.shape[0]
    period = np.random.uniform(3, 22); phase = np.random.uniform(0, 2*np.pi)
    b = (1 + amp*np.sin(2*np.pi*np.arange(h)/period + phase)).astype(np.float32)
    return _clip(x * (b[:, None, None] if x.ndim == 3 else b[:, None]))

# ------------------------------------------------------------- photometric --
def illumination_gradient(x, s):
    st = [0.10, 0.20, 0.32, 0.46, 0.62][s-1]; h, w = x.shape[:2]
    ang = np.random.uniform(0, 2*np.pi); yy, xx = np.mgrid[0:h, 0:w]
    u = (xx/w-.5)*np.cos(ang) + (yy/h-.5)*np.sin(ang)
    g = (1 + st*u/(np.abs(u).max()+1e-8)).astype(np.float32)
    return _clip(x * _mask3(g, x))

def brightness_drift(x, s):
    # v2: gamma instead of an additive offset. Gamma maps [0,1] -> [0,1] and CANNOT clip.
    # v1 saturated 36.8% of pixels at severity 5 on Magnetic Tile, which destroyed dynamic
    # range rather than shifting brightness and made the top of the ladder meaningless.
    g = [1.12, 1.26, 1.45, 1.72, 2.05][s-1]
    if np.random.rand() < 0.5: g = 1.0/g
    return _clip(np.power(_clip(x), g))

def contrast_loss(x, s):
    g = [0.80, 0.65, 0.50, 0.36, 0.24][s-1]
    m = x.mean(axis=(0, 1), keepdims=True)
    return _clip((x-m)*g + m)

# ---------------------------------------------------------------- pipeline --
def jpeg_compression(x, s):
    q = [70, 50, 32, 18, 9][s-1]
    src = (x[..., ::-1]*255).astype(np.uint8) if x.ndim == 3 else (x*255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", src, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR if x.ndim == 3 else cv2.IMREAD_GRAYSCALE)
    return (dec[..., ::-1] if x.ndim == 3 else dec).astype(np.float32)/255.

CORRUPTIONS = {
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "vignetting": vignetting,
    "lens_contamination": lens_contamination, "vibration_jitter": vibration_jitter,
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise,
    "scanline_banding": scanline_banding, "illumination_gradient": illumination_gradient,
    "brightness_drift": brightness_drift, "contrast_loss": contrast_loss,
    "jpeg": jpeg_compression,
}
TRAIN_FAMILIES = ["defocus_blur", "gaussian_noise", "illumination_gradient",
                  "jpeg", "lens_contamination", "scanline_banding"]
TEST_FAMILIES  = ["motion_blur", "shot_noise", "brightness_drift",
                  "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]
CONDITIONS = [("clean", 0)] + [(f, s) for f in CORRUPTIONS for s in SEVERITIES]


def corruption_seed(image_id, family, severity=None):
    # Depends on (image_id, family) only, NOT severity: nuisance parameters (blur angle,
    # banding period, blob positions, gradient orientation) stay fixed so that severity is
    # the sole varying factor. Seeding on severity too made scanline_banding score 0.20.
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return int.from_bytes(h[:4], "little")


def apply_corruption(img_u8, family, severity, image_id=None, seed=None):
    if family == "clean":
        return img_u8
    if seed is None and image_id is not None:
        seed = corruption_seed(image_id, family)
    st = None
    if seed is not None:
        st = np.random.get_state(); np.random.seed(seed % (2**32))
    try:
        out = CORRUPTIONS[family](img_u8.astype(np.float32)/255., severity)
    finally:
        if st is not None: np.random.set_state(st)
    return (np.clip(out, 0, 1)*255).astype(np.uint8)


# ------------------------------------------------------- quality descriptors -
def _gray(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im
    return g.astype(np.float32)/255.

def _sharp(im): return float(cv2.Laplacian(_gray(im), cv2.CV_32F).var())

def _noise(im):
    g = _gray(im); h, w = g.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.abs(cv2.filter2D(g, -1, M)).sum()*math.sqrt(math.pi/2) /
                 (6*max(w-2, 1)*max(h-2, 1)))

def _block(im):
    dh = np.abs(np.diff(_gray(im), axis=1))
    on = dh[:, 7::8].mean() if dh.shape[1] > 8 else 0.0
    return float(on/(dh.mean()+1e-8) - 1.0)

def _hf(im):
    g = _gray(im); f = np.abs(np.fft.fftshift(np.fft.fft2(g))); h, w = g.shape
    cy, cx = h//2, w//2; r = max(4, min(h, w)//8)
    return float(1.0 - f[cy-r:cy+r, cx-r:cx+r].sum()/(f.sum()+1e-8))


def quality_descriptor(img_u8):
    # 8-d ABSOLUTE no-reference descriptor (unchanged from v1).
    g = _gray(img_u8); h, w = g.shape
    if img_u8.ndim == 3:
        rg = img_u8[..., 0].astype(np.float32) - img_u8[..., 1]
        yb = .5*(img_u8[..., 0].astype(np.float32) + img_u8[..., 1]) - img_u8[..., 2]
        colour = float((np.sqrt(rg.std()**2 + yb.std()**2)
                        + .3*np.sqrt(rg.mean()**2 + yb.mean()**2))/255.)
    else:
        colour = 0.0
    cen = g[h//4:3*h//4, w//4:3*w//4]
    per = (g.sum()-cen.sum())/max(g.size-cen.size, 1)
    v = np.array([math.log1p(max(_sharp(img_u8), 0.)*1e3), _hf(img_u8),
                  math.log1p(max(_noise(img_u8), 0.)*1e3), _block(img_u8),
                  float(g.mean()), float(g.std()), colour,
                  float(cen.mean()/(per+1e-8))], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


def quality_descriptor_relative(img_u8):
    # 8-d PERTURBATION-RESPONSE descriptor: how much does a statistic move when a known
    # perturbation is applied? Higher severity signal than the absolute set (0.324 vs 0.236
    # on the synthetic benchmark) but NO reduction in class leakage on its own -- ratios
    # remove the absolute texture level, not the spectral shape. Use with class-conditional
    # standardisation, never alone.
    g8 = (_gray(img_u8)*255).astype(np.uint8); eps = 1e-8
    b = cv2.GaussianBlur(g8, (0, 0), 1.5)
    dn = cv2.resize(cv2.resize(g8, (max(2, g8.shape[1]//2), max(2, g8.shape[0]//2)),
                               interpolation=cv2.INTER_AREA),
                    (g8.shape[1], g8.shape[0]), interpolation=cv2.INTER_LINEAR)
    rn = np.random.default_rng(12345)
    nz = np.clip(g8.astype(np.float32) + rn.normal(0, 12, g8.shape), 0, 255).astype(np.uint8)
    _, e = cv2.imencode(".jpg", g8, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
    jp = cv2.imdecode(e, cv2.IMREAD_GRAYSCALE)
    kh = np.zeros((9, 9), np.float32); kh[4, :] = 1/9.
    hb = cv2.filter2D(g8.astype(np.float32), -1, kh).astype(np.uint8)
    vb = cv2.filter2D(g8.astype(np.float32), -1, kh.T).astype(np.uint8)
    x = g8.astype(np.float32)/255.
    v = np.array([
        math.log((_sharp(g8)+eps)/(_sharp(b)+eps)),
        math.log((_sharp(g8)+eps)/(_sharp(dn)+eps)),
        math.log((_noise(nz)+eps)/(_noise(g8)+eps)),
        _block(jp) - _block(g8),
        float(((x+0.25) > 1.0).mean() + ((x-0.25) < 0.0).mean()),
        abs(math.log((_sharp(hb)+eps)/(_sharp(vb)+eps))),
        math.log((_hf(g8)+eps)/(_hf(b)+eps)),
        math.log((_gray(g8).std()+eps)/(_gray(b).std()+eps)),
    ], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


QUALITY_NAMES = ["sharpness", "hf_energy", "noise", "blockiness",
                 "luminance_mean", "luminance_std", "colourfulness", "vignette_ratio"]
RELATIVE_NAMES = ["blur_headroom", "resolution_headroom", "noise_headroom",
                  "compression_headroom", "clipping_headroom", "blur_anisotropy",
                  "hf_retention", "contrast_retention"]
QUALITY_DIM = 8


class ClassConditionalStandardiser:
    # z = (q - mu_c) / sigma_c, with mu_c and sigma_c estimated on DEVELOPMENT data only.
    #
    # Rationale: degradation is relative. A blurry-looking crazing image and a sharp-looking
    # patches image can have identical absolute sharpness; what makes one degraded is that it
    # is blurrier than crazing images normally are. Removing the class-conditional mean strips
    # the content component and leaves the deviation-from-typical -- which is the degradation.
    #
    # At test time c is the model's own prediction. There is no feedback loop: a per-image
    # scalar temperature cannot change the argmax, so the prediction is fixed before the
    # calibrator runs.
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.mu = None; self.sd = None

    def fit(self, q, y):
        d = q.shape[1]
        self.mu = np.zeros((self.n_classes, d), np.float32)
        self.sd = np.ones((self.n_classes, d), np.float32)
        gm, gs = q.mean(0), q.std(0) + 1e-6
        for c in range(self.n_classes):
            m = (y == c)
            if m.sum() >= 5:                 # fall back to global stats for tiny classes
                self.mu[c] = q[m].mean(0); self.sd[c] = q[m].std(0) + 1e-6
            else:
                self.mu[c] = gm; self.sd[c] = gs
        return self

    def transform(self, q, y_pred):
        y_pred = np.asarray(y_pred).astype(int)
        return ((q - self.mu[y_pred]) / self.sd[y_pred]).astype(np.float32)
"""
print(f"embedded core module: {len(CORE_V2.splitlines())} lines")

embedded core module: 246 lines


## 1. Setup and feature extraction

Same seed, subset, backbone and conditions as Phase 0.6/0.7 so numbers remain comparable.

In [ ]:
#@title Dependencies and core module
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11","scikit-learn==1.5.2",
                "opencv-python-headless==4.10.0.84","pandas==2.2.3","kagglehub","tqdm"],check=True)

import os, json, math, time, random, hashlib, warnings
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")

SEED=20260821
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
ROOT=Path("/content/sdic"); OUT=ROOT/"phase0"; OUT.mkdir(parents=True,exist_ok=True)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

sys.path.insert(0,str(ROOT))
_t=ROOT/"sdic_core_v2.py"; _w=hashlib.sha256(CORE_V2.encode()).hexdigest()
if not _t.exists() or hashlib.sha256(_t.read_text().encode()).hexdigest()!=_w:
    _t.write_text(CORE_V2); print(f"sdic_core_v2.py written, sha256 {_w[:16]}")
else: print(f"sdic_core_v2.py matches audited copy, sha256 {_w[:16]}")
import importlib, sdic_core_v2; importlib.reload(sdic_core_v2)
from sdic_core_v2 import (CORRUPTIONS, TRAIN_FAMILIES, TEST_FAMILIES, SEVERITIES,
                          apply_corruption, quality_descriptor, QUALITY_DIM)
print("device:", DEVICE)

sdic_core_v2.py written, sha256 fd8e3c534562f9d2
device: cuda


In [ ]:
#@title Index NEU and extract features (the only expensive cell)
import kagglehub
from timm.data import resolve_model_data_config
_INTERP={"bilinear":cv2.INTER_LINEAR,"bicubic":cv2.INTER_CUBIC,
         "nearest":cv2.INTER_NEAREST,"area":cv2.INTER_AREA}
NEU_ROOT=Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
neu=[p for p in sorted(NEU_ROOT.rglob("*.jpg"))
     if p.parent.name.lower() not in {"train","validation","images","annotations"}]
labels=np.array([p.parent.name for p in neu])
LUT={n:i for i,n in enumerate(sorted(set(labels)))}
y_all=np.array([LUT[l] for l in labels]); NC=len(LUT)
def rd(p): return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)

BACKBONE="resnet50.a1_in1k"; N_PER_CLASS=150
set_seed()
per=defaultdict(list)
for p,yy in zip(neu,y_all): per[yy].append(p)
sel=[]
for c,v in per.items():
    pick=np.random.default_rng(SEED+c).choice(len(v),min(N_PER_CLASS,len(v)),replace=False)
    sel += [neu.index(v[t]) for t in pick]
sel=np.array(sorted(sel)); S_paths=[neu[i] for i in sel]; S_y=y_all[sel]
print(f"{len(S_paths)} images, {NC} classes")

model=timm.create_model(BACKBONE,pretrained=True,num_classes=0).eval().to(DEVICE)
_c=resolve_model_data_config(model)
PP={"mean":np.array(_c["mean"],np.float32),"std":np.array(_c["std"],np.float32),"size":224,
    "interp":_INTERP.get(_c["interpolation"],cv2.INTER_CUBIC),
    "crop_pct":float(_c.get("crop_pct") or 1.0)}
def prep(im):
    sz=PP["size"]; to=int(round(sz/PP["crop_pct"])); h,w=im.shape[:2]; s=to/min(h,w)
    r=cv2.resize(im,(max(1,int(round(w*s))),max(1,int(round(h*s)))),interpolation=PP["interp"])
    hh,ww=r.shape[:2]; t,l=(hh-sz)//2,(ww-sz)//2
    x=(r[t:t+sz,l:l+sz].astype(np.float32)/255.-PP["mean"])/PP["std"]
    return torch.from_numpy(x).permute(2,0,1)

# FIX: Ensure all severities are cached for TRAIN_FAMILIES
CONDS=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in SEVERITIES] \
                   +[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]

@torch.no_grad()
def extract(fam,sev,bs=96):
    Fs,Qs=[],[]
    for i in range(0,len(S_paths),bs):
        xs,qs=[],[]
        for p in S_paths[i:i+bs]:
            im=apply_corruption(rd(p),fam,max(sev,1),image_id=p.stem) if fam!="clean" else rd(p)
            qs.append(quality_descriptor(im)); xs.append(prep(im))
        with torch.autocast("cuda",enabled=DEVICE=="cuda"):
            Fs.append(model(torch.stack(xs).to(DEVICE)).float().cpu().numpy())
        Qs.append(np.stack(qs))
    return np.concatenate(Fs),np.concatenate(Qs)

t0=time.time(); CACHE={}
for fam,sev in tqdm(CONDS,desc="extract"): CACHE[(fam,sev)]=extract(fam,sev)
del model; torch.cuda.empty_cache()
print(f"{len(CONDS)} conditions in {(time.time()-t0)/60:.1f} min")
print("Every image is now cached under every condition, so all five folds are free.")


Using Colab cache for faster access to the 'neu-surface-defect-database' dataset.
900 images, 6 classes


extract:   0%|          | 0/61 [00:00<?, ?it/s]

61 conditions in 9.4 min
Every image is now cached under every condition, so all five folds are free.


## 2. Standardiser, temperature heads, metrics

In [ ]:
class RobustClassStandardiser:
    """z = clip((q - mu_c)/sigma_c, +/- clip_z). The variance floor is what stopped the
    collapse: without it a near-constant class-dimension produced |z| in the hundreds of
    thousands, and fixing only the fit distribution was not enough (C6b still collapsed)."""
    def __init__(self,n_classes,sd_floor=0.25,clip_z=5.0):
        self.n=n_classes; self.sd_floor=sd_floor; self.clip_z=clip_z
    def fit(self,q,y):
        gm,gs=q.mean(0),q.std(0)+1e-6
        self.mu=np.tile(gm,(self.n,1)).astype(np.float32)
        self.sd=np.tile(gs,(self.n,1)).astype(np.float32)
        for c in range(self.n):
            m=(y==c)
            if m.sum()>=20:
                self.mu[c]=q[m].mean(0)
                self.sd[c]=np.maximum(q[m].std(0),self.sd_floor*gs)+1e-6
        return self
    def transform(self,q,y_pred):
        y_pred=np.asarray(y_pred).astype(int)
        return np.clip((q-self.mu[y_pred])/self.sd[y_pred],
                       -self.clip_z,self.clip_z).astype(np.float32)

EPS=1e-2
class ScalarT(nn.Module):
    def __init__(self,d=None):
        super().__init__(); self.log_t=nn.Parameter(torch.zeros(()))
    def temperature(self,q): return self.log_t.exp().expand(q.shape[0])+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class LinearT(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        nn.init.constant_(self.lin.bias,math.log(math.exp(1.0-EPS)-1.0))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q): return F.softplus(self.lin((q-self.mu)/self.sd).squeeze(-1))+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class BoundedT(nn.Module):
    """T = T_lo + (T_hi - T_lo)*sigmoid(w.z + b): collapse is structurally impossible."""
    def __init__(self,d,t_lo=0.5,t_hi=5.0):
        super().__init__(); self.t_lo,self.t_hi=t_lo,t_hi
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        p=(1.0-t_lo)/(t_hi-t_lo); nn.init.constant_(self.lin.bias,math.log(p/(1-p)))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q):
        return self.t_lo+(self.t_hi-self.t_lo)*torch.sigmoid(
            self.lin((q-self.mu)/self.sd).squeeze(-1))
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class MLPT(LinearT):
    def __init__(self,d,h=32):
        super().__init__(d); self.lin=None
        self.net=nn.Sequential(nn.Linear(d,h),nn.SiLU(),nn.Linear(h,h//2),nn.SiLU(),
                               nn.Linear(h//2,1))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias,math.log(math.exp(1.0-EPS)-1.0))
    def temperature(self,q): return F.softplus(self.net((q-self.mu)/self.sd).squeeze(-1))+EPS

def fit_cal(cls,L,Q,Y,epochs=400,lr=1e-2,seed=SEED):
    set_seed(seed)
    L=torch.as_tensor(L,dtype=torch.float32); Q=torch.as_tensor(Q,dtype=torch.float32)
    Y=torch.as_tensor(Y,dtype=torch.long)
    m=cls(Q.shape[1])
    if hasattr(m,"fit_norm"): m.fit_norm(Q)
    opt=torch.optim.Adam(m.parameters(),lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(m(L,Q),Y).backward(); opt.step()
    return m.eval()

def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece_em(p,y,nb=15):
    conf,pred=p.max(1),p.argmax(1); corr=(pred==y).astype(float); o=np.argsort(conf)
    return float(sum(len(c)/len(y)*abs(corr[c].mean()-conf[c].mean())
                     for c in np.array_split(o,nb) if len(c)))
def nll_pi(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None))
def aurc(p,y):
    conf,pred=p.max(1),p.argmax(1); e=(pred!=y).astype(float)[np.argsort(-conf)]
    n=len(e); return float(np.trapezoid(np.cumsum(e)/np.arange(1,n+1),np.arange(1,n+1)/n))

for cls in (LinearT,BoundedT,MLPT):
    m=cls(8); m.fit_norm(torch.randn(200,8))
    T=m.temperature(torch.randn(200,8)).detach()
    assert abs(T.mean().item()-1.0)<1e-4 and T.std().item()<1e-5, cls.__name__
print("all heads initialise to T = 1.0 exactly")

all heads initialise to T = 1.0 exactly


## 3. Five folds, with selection on a held-out split

Per fold: 60% probe training and standardiser reference, 20% calibrator fitting, 20% **selection**,
20% test. The selection split shares no images with either the calibrator fit or the test fold,
and the test fold is read exactly once, at the end.

Selection is run twice — under train families (strict, the primary) and under test families
(optimistic, a sensitivity check) — so any disagreement between them is visible rather than
hidden.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split

# NESTED FAMILY HOLDOUT. The six train families are split: four fit the calibrators, two are
# held out for selection. Selection therefore measures generalisation to UNSEEN FAMILIES on
# UNSEEN IMAGES -- the same kind of shift the test set measures -- without ever touching the
# six reported test families.
CAL_FAMILIES = TRAIN_FAMILIES[:4]
SEL_FAMILIES = TRAIN_FAMILIES[4:]
CAL_CONDS       =[("clean",0)]+[(f,s) for f in CAL_FAMILIES for s in (1,3,5)]
SEL_CONDS_NESTED=[(f,s) for f in SEL_FAMILIES  for s in SEVERITIES]   # primary
SEL_CONDS_STRICT=[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)]    # v0.9 rule, kept to expose it
SEL_CONDS_OPTIM =[(f,s) for f in TEST_FAMILIES  for s in SEVERITIES] # compromised, diagnostic only
TEST_CONDS      =[(f,s) for f in TEST_FAMILIES  for s in SEVERITIES]
print(f"calibration families {CAL_FAMILIES}\nselection families   {SEL_FAMILIES}\n"
      f"test families        {TEST_FAMILIES}")

ARM_SPECS={
 "C2 scalar / augmented":      ("scalar",ScalarT, "q"),
 "C1 scalar / clean":          ("clean", ScalarT, "q"),
 "C3 MLP / raw q":             ("scalar",MLPT,    "q"),
 "C4 linear / raw q":          ("scalar",LinearT, "q"),
 "C5 linear / class one-hot":  ("scalar",LinearT, "h"),
 "C6c linear / std (robust)":  ("scalar",LinearT, "z"),
 "C7 bounded / std (robust)":  ("scalar",BoundedT,"z"),
 "C7q bounded / raw q":        ("scalar",BoundedT,"q"),
 "C8 linear / class + q":      ("scalar",LinearT, "hq"),
 "C8b bounded / class + q":    ("scalar",BoundedT,"hq"),
}
QUALITY_ARMS=[k for k in ARM_SPECS if k.startswith(("C4","C6","C7","C8"))]

def gather(idx,conds,logit_fn,std=None):
    L,Q,Y,IM=[],[],[],[]
    for fam,sev in conds:
        Fx,Qx=CACHE[(fam,sev)]
        L.append(logit_fn(Fx[idx])); Q.append(Qx[idx]); Y.append(S_y[idx]); IM.append(idx)
    L=np.concatenate(L); Q=np.concatenate(Q); Y=np.concatenate(Y); IM=np.concatenate(IM)
    H=np.zeros((len(L),NC),np.float32); H[np.arange(len(L)),L.argmax(1)]=1.0
    Z=std.transform(Q,L.argmax(1)) if std is not None else None
    HQ=np.concatenate([H,Q],1).astype(np.float32)      # C8: class offset + quality slope
    return dict(L=L,q=Q,h=H,z=Z,hq=HQ,y=Y,im=IM)

def evaluate(m,pk,feat):
    X=torch.as_tensor(pk[feat],dtype=torch.float32)
    with torch.no_grad():
        out=m(torch.as_tensor(pk["L"],dtype=torch.float32),X).numpy()
        T=m.temperature(X).numpy()
    p=softmax(out)
    return dict(nll=float(nll_pi(p,pk["y"]).mean()),ece=ece_em(p,pk["y"]),
                aurc=aurc(p,pk["y"]),acc=float((p.argmax(1)==pk["y"]).mean()),
                T_min=float(T.min()),T_max=float(T.max())), p, nll_pi(p,pk["y"])

def sound(r):
    return not ((r["T_min"]-EPS<0.1*EPS and r["T_min"]<1.0) or r["T_max"]>50)

skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
FOLDS=[te for _,te in skf.split(np.zeros(len(S_y)),S_y)]
Fc,Qc=CACHE[("clean",0)]

fold_rows=[]; sel_rows=[]; fam_rows=[]
POOL =defaultdict(lambda: dict(nll=[],y=[],im=[],p=[]))
POOLF=defaultdict(lambda: dict(nll=[],y=[],im=[],p=[]))
for k in range(5):
    te=FOLDS[k]; rest=np.concatenate([FOLDS[j] for j in range(5) if j!=k])
    tr,tmp=train_test_split(rest,test_size=0.4,random_state=SEED+k,stratify=S_y[rest])
    va_fit,va_sel=train_test_split(tmp,test_size=0.5,random_state=SEED+k,stratify=S_y[tmp])

    probe=make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=4000,class_weight="balanced"))
    probe.fit(Fc[tr],S_y[tr])
    def lg_of(Fx,_p=probe):
        d=_p.decision_function(Fx); return d if d.ndim>1 else np.stack([-d,d],1)

    dev=gather(tr,CAL_CONDS,lg_of)
    STD=RobustClassStandardiser(NC).fit(dev["q"],dev["y"])
    P_fit  =gather(va_fit,CAL_CONDS,lg_of,STD)
    P_fitc =gather(va_fit,[("clean",0)],lg_of,STD)
    P_selN =gather(va_sel,SEL_CONDS_NESTED,lg_of,STD)
    P_selS =gather(va_sel,SEL_CONDS_STRICT,lg_of,STD)
    P_selO =gather(va_sel,SEL_CONDS_OPTIM ,lg_of,STD)
    P_te   =gather(te,TEST_CONDS,lg_of,STD)
    P_te_by_fam={f:gather(te,[(f,sv) for sv in SEVERITIES],lg_of,STD) for f in TEST_FAMILIES}

    for name,(mode,cls,feat) in ARM_SPECS.items():
        src = P_fitc if mode=="clean" else P_fit
        m = fit_cal(cls, src["L"], src[feat], src["y"])
        rte,pte,pim = evaluate(m,P_te,feat)
        rsn,_,_ = evaluate(m,P_selN,feat)
        rss,_,_ = evaluate(m,P_selS,feat)
        rso,_,_ = evaluate(m,P_selO,feat)
        fold_rows.append(dict(fold=k,arm=name,**rte,sound=sound(rte)))
        sel_rows.append(dict(fold=k,arm=name,
                             sel_nested_nll=rsn["nll"],sel_nested_ece=rsn["ece"],
                             sel_strict_nll=rss["nll"],sel_strict_ece=rss["ece"],
                             sel_optim_nll=rso["nll"],sel_optim_ece=rso["ece"],
                             sel_sound=sound(rsn) and sound(rss)))
        POOL[name]["nll"].append(pim); POOL[name]["y"].append(P_te["y"])
        POOL[name]["im"].append(P_te["im"]); POOL[name]["p"].append(pte)
        for f,pk in P_te_by_fam.items():
            rf,pf,pimf = evaluate(m,pk,feat)
            fam_rows.append(dict(fold=k,arm=name,family=f,**rf))
            POOLF[(name,f)]["nll"].append(pimf); POOLF[(name,f)]["y"].append(pk["y"])
            POOLF[(name,f)]["im"].append(pk["im"]); POOLF[(name,f)]["p"].append(pf)

    p0=softmax(P_te["L"])
    fold_rows.append(dict(fold=k,arm="C0 uncalibrated",nll=float(nll_pi(p0,P_te["y"]).mean()),
                          ece=ece_em(p0,P_te["y"]),aurc=aurc(p0,P_te["y"]),
                          acc=float((p0.argmax(1)==P_te["y"]).mean()),T_min=1.,T_max=1.,sound=True))
    POOL["C0 uncalibrated"]["nll"].append(nll_pi(p0,P_te["y"]))
    POOL["C0 uncalibrated"]["y"].append(P_te["y"]); POOL["C0 uncalibrated"]["im"].append(P_te["im"])
    POOL["C0 uncalibrated"]["p"].append(p0)
    print(f"  fold {k} done")

FR=pd.DataFrame(fold_rows); SR=pd.DataFrame(sel_rows); FAM=pd.DataFrame(fam_rows)
for d in list(POOL.values())+list(POOLF.values()):
    for kk in ("nll","y","im","p"): d[kk]=np.concatenate(d[kk])
print(f"\n{len(FR)} arm-fold results | pooled test rows per arm: {len(POOL['C4 linear / raw q']['y'])}")

calibration families ['defocus_blur', 'gaussian_noise', 'illumination_gradient', 'jpeg']
selection families   ['lens_contamination', 'scanline_banding']
test families        ['motion_blur', 'shot_noise', 'brightness_drift', 'contrast_loss', 'vignetting', 'vibration_jitter']
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done

55 arm-fold results | pooled test rows per arm: 27000


## 4. Which arm does validation choose?

In [ ]:
UNCAL=FR[FR.arm=="C0 uncalibrated"].nll.mean()
agg=(FR.groupby("arm").agg(nll_mean=("nll","mean"),nll_sd=("nll","std"),
                           ece_mean=("ece","mean"),ece_sd=("ece","std"),
                           aurc_mean=("aurc","mean"),T_min=("T_min","min"),
                           n_sound=("sound","sum")).round(4))
# C0 is the reference; comparing it against itself produced a rounding artefact in v0.8
# (agg.nll_mean is rounded to 4dp, UNCAL is not, so 1.1255 > 1.125494 came out True).
agg["gate"]=np.where(agg.index=="C0 uncalibrated","reference",
             np.where(agg.n_sound<5,"FAILS in >=1 fold",
             np.where(agg.nll_mean>UNCAL+1e-6,"worse than uncalibrated","sound")))
display(agg.sort_values("nll_mean"))
agg.to_csv(OUT/"phase0_11_arms.csv")

sa=SR.groupby("arm").agg(nested_nll=("sel_nested_nll","mean"),nested_ece=("sel_nested_ece","mean"),
                         strict_nll=("sel_strict_nll","mean"),strict_ece=("sel_strict_ece","mean"),
                         optim_nll=("sel_optim_nll","mean"),optim_ece=("sel_optim_ece","mean"),
                         n_sound=("sel_sound","sum")).round(4)
elig=[a for a in QUALITY_ARMS if a in sa.index and sa.loc[a,"n_sound"]==5
      and agg.loc[a,"gate"]=="sound"]
print("\nSELECTION (test fold never consulted)")
display(sa.loc[[a for a in QUALITY_ARMS]])
from scipy.stats import spearmanr
if elig:
    W_NLL = min(elig,key=lambda a: sa.loc[a,"nested_nll"])
    W_ECE = min(elig,key=lambda a: sa.loc[a,"nested_ece"])
    print(f"\n  winner on NESTED validation NLL : {W_NLL}")
    print(f"  winner on NESTED validation ECE : {W_ECE}")
    if W_NLL!=W_ECE:
        print("  the two metrics select different arms; both are carried forward")
    WINNERS=sorted({W_NLL,W_ECE})

    # Does each selection rule actually predict test performance?
    print("\n  Does the selection signal predict the test outcome? (Spearman over eligible arms)")
    for rule,col in [("nested (4 fit / 2 held-out families)","nested_nll"),
                     ("strict  (all train families)","strict_nll"),
                     ("optimistic (test families)","optim_nll")]:
        rho=spearmanr([sa.loc[a,col] for a in elig],
                      [agg.loc[a,"nll_mean"] for a in elig]).statistic
        flag="" if rho>=0.6 else "   <- WEAK: this rule does not predict the test outcome"
        print(f"    {rule:38s} rho = {rho:+.3f}{flag}")
    RHO_BEST=max(spearmanr([sa.loc[a,c] for a in elig],
                           [agg.loc[a,"nll_mean"] for a in elig]).statistic
                 for c in ("nested_nll","strict_nll"))
    print(f"  best legitimate rule: rho = {RHO_BEST:+.3f}"
          f"  ({'usable' if RHO_BEST>=0.6 else 'NOT usable to choose the method'})")
    print("  (the optimistic rule is excluded here: it selects on the reported test families)")
else:
    WINNERS=[]; RHO_BEST=float("nan")
    print("\nNo quality-conditioned arm is sound in all five folds.")

,nll_mean,nll_sd,ece_mean,ece_sd,aurc_mean,T_min,n_sound,gate
arm,,,,,,,,
C7q bounded / raw q,0.7630,0.0835,0.0329,0.0045,0.0684,0.5006,5,sound
C2 scalar / augmented,0.7793,0.0250,0.0382,0.0115,0.0948,2.1291,5,sound
C4 linear / raw q,0.7885,0.1803,0.0331,0.0070,0.0692,0.0195,5,sound
C8b bounded / class + q,0.8213,0.1341,0.0378,0.0132,0.0693,0.5054,5,sound
C5 linear / class one-hot,0.8629,0.1355,0.0432,0.0089,0.0872,0.4166,5,sound
C7 bounded / std (robust),0.8690,0.1347,0.0406,0.0084,0.0751,0.5060,5,sound
C8 linear / class + q,0.9448,0.2577,0.0433,0.0157,0.0747,0.0241,5,sound
C6c linear / std (robust),1.0304,0.2970,0.0470,0.0110,0.0896,0.0248,5,sound
C1 scalar / clean,1.0786,0.1804,0.1122,0.0188,0.0884,0.9062,5,sound



SELECTION (test fold never consulted)


,nested_nll,nested_ece,strict_nll,strict_ece,optim_nll,optim_ece,n_sound
arm,,,,,,,
C4 linear / raw q,2.3300,0.2103,1.2314,0.0740,0.8356,0.0345,5
C6c linear / std (robust),2.3124,0.2119,1.2220,0.0778,1.0885,0.0482,5
C7 bounded / std (robust),2.2193,0.2166,1.1832,0.0808,0.9236,0.0430,5
C7q bounded / raw q,2.1767,0.2112,1.1703,0.0742,0.8132,0.0333,5
C8 linear / class + q,2.1791,0.1944,1.1506,0.0709,1.0175,0.0416,5
C8b bounded / class + q,1.9813,0.1949,1.0773,0.0731,0.8788,0.0357,5



  winner on NESTED validation NLL : C8b bounded / class + q
  winner on NESTED validation ECE : C8 linear / class + q
  the two metrics select different arms; both are carried forward

  Does the selection signal predict the test outcome? (Spearman over eligible arms)
    nested (4 fit / 2 held-out families)   rho = +0.257   <- WEAK: this rule does not predict the test outcome
    strict  (all train families)           rho = +0.029   <- WEAK: this rule does not predict the test outcome
    optimistic (test families)             rho = +1.000
  best legitimate rule: rho = +0.257  (NOT usable to choose the method)
  (the optimistic rule is excluded here: it selects on the reported test families)


## 5. Two kinds of uncertainty, and only one of them was being reported

Every interval so far resampled **images inside a fixed set of six test families**. That answers
*"would this hold on different images of these six corruptions?"* It does not answer *"would this
hold on six different corruptions?"* — which is the question the paper actually asks.

The Phase 0.10 run showed those are wildly different questions. The two held-out selection
families produced NLL around 2.2–2.4 while the six test families produced 0.76–1.03: roughly
2.5× harder. Corruption families are **not exchangeable**, so treating six of them as a large
sample is wrong. The effective sample size for a claim about unseen corruption is closer to
**n = 6** than to n = 5,400.

Both intervals are therefore computed for every comparison:

* **image-level** — resamples images, conditional on this family set. Narrow. Reported so far.
* **family-level** — computes the per-family difference, then a *t* interval over the six
  families with 5 degrees of freedom. This is the one that supports a generalisation claim.

A method that wins on the image interval but not the family interval has been shown to work on
*these six corruptions*, not on corruption in general. That distinction has to reach the paper.

In [ ]:
from scipy.stats import t as _t

def image_boot_nll(a,b,ims,n_boot=10000,seed=SEED):
    rng=np.random.default_rng(seed); d=a-b
    uniq=np.unique(ims); by={u:np.where(ims==u)[0] for u in uniq}
    bs=np.array([d[np.concatenate([by[u] for u in rng.choice(uniq,len(uniq),True)])].mean()
                 for _ in range(n_boot)])
    return float(d.mean()),float(np.quantile(bs,.025)),float(np.quantile(bs,.975))

def family_ci(per_family_deltas,alpha=0.05):
    v=np.asarray(per_family_deltas,float); n=len(v)
    m=float(v.mean()); sd=float(v.std(ddof=1))
    h=_t.ppf(1-alpha/2,n-1)*sd/math.sqrt(n)
    return m,m-h,m+h,sd,int(n)

def compare(A,B):
    dn_f=[float(POOLF[(A,f)]["nll"].mean()-POOLF[(B,f)]["nll"].mean()) for f in TEST_FAMILIES]
    de_f=[float(ece_em(POOLF[(A,f)]["p"],POOLF[(A,f)]["y"])
                -ece_em(POOLF[(B,f)]["p"],POOLF[(B,f)]["y"])) for f in TEST_FAMILIES]
    dn,ilo,ihi=image_boot_nll(POOL[A]["nll"],POOL[B]["nll"],POOL[A]["im"])
    fm,flo,fhi,fsd,n=family_ci(dn_f)
    em,elo,ehi,esd,_=family_ci(de_f)
    return dict(arm=A,vs=B,
        d_nll=round(dn,4), nll_image_ci=f"[{ilo:.4f}, {ihi:.4f}]", win_nll_image=bool(ihi<0),
        nll_family_ci=f"[{flo:.4f}, {fhi:.4f}]", win_nll_family=bool(fhi<0),
        nll_family_sd=round(fsd,4),
        d_ece=round(em,4), ece_family_ci=f"[{elo:.4f}, {ehi:.4f}]", win_ece_family=bool(ehi<0),
        ece_family_sd=round(esd,4),
        image_ci_width=round(ihi-ilo,5), family_ci_width=round(fhi-flo,5),
        per_family_nll=[round(x,3) for x in dn_f])

REF=["C2 scalar / augmented","C5 linear / class one-hot"]
SOUND_Q=[a for a in QUALITY_ARMS if agg.loc[a,"gate"]=="sound"]
T=pd.DataFrame([compare(A,B) for A in SOUND_Q for B in REF])
display(T[["arm","vs","d_nll","nll_image_ci","win_nll_image","nll_family_ci","win_nll_family",
           "image_ci_width","family_ci_width"]])
T.to_csv(OUT/"phase0_11_bootstrap.csv",index=False)

wr=(T.family_ci_width/T.image_ci_width).replace([np.inf,-np.inf],np.nan).dropna()
print(f"\nfamily interval / image interval width: min {wr.min():.1f}x  "
      f"median {wr.median():.1f}x  max {wr.max():.1f}x")
flips=T[(T.win_nll_image) & (~T.win_nll_family)]
if len(flips):
    print(f"\n{len(flips)} comparison(s) significant on images but NOT across families:")
    for _,r in flips.iterrows():
        print(f"  {r['arm']} vs {r['vs'].split(' ')[0]}: per-family deltas {r['per_family_nll']}")
    print("  These are claims about these six corruptions, not about corruption in general.")

,arm,vs,d_nll,nll_image_ci,win_nll_image,nll_family_ci,win_nll_family,image_ci_width,family_ci_width
0,C4 linear / raw q,C2 scalar / augmented,0.0091,"[-0.0096, 0.0289]",False,"[-0.2107, 0.2290]",False,0.03848,0.43973
1,C4 linear / raw q,C5 linear / class one-hot,-0.0745,"[-0.1012, -0.0489]",True,"[-0.3765, 0.2276]",False,0.05225,0.60411
2,C6c linear / std (robust),C2 scalar / augmented,0.2511,"[0.2137, 0.2887]",False,"[-0.6016, 1.1038]",False,0.07497,1.70538
3,C6c linear / std (robust),C5 linear / class one-hot,0.1675,"[0.1321, 0.2023]",False,"[-0.7489, 1.0839]",False,0.07015,1.83280
4,C7 bounded / std (robust),C2 scalar / augmented,0.0897,"[0.0703, 0.1097]",False,"[-0.3626, 0.5420]",False,0.03935,0.90458
5,C7 bounded / std (robust),C5 linear / class one-hot,0.0061,"[-0.0235, 0.0354]",False,"[-0.5228, 0.5350]",False,0.05884,1.05782
6,C7q bounded / raw q,C2 scalar / augmented,-0.0164,"[-0.0278, -0.0049]",True,"[-0.1720, 0.1393]",False,0.02284,0.31123
7,C7q bounded / raw q,C5 linear / class one-hot,-0.0999,"[-0.1263, -0.0739]",True,"[-0.3467, 0.1468]",False,0.05247,0.49357
8,C8 linear / class + q,C2 scalar / augmented,0.1655,"[0.1181, 0.2149]",False,"[-0.1693, 0.5002]",False,0.09683,0.66950
9,C8 linear / class + q,C5 linear / class one-hot,0.0819,"[0.0574, 0.1079]",False,"[-0.1415, 0.3052]",False,0.05050,0.44669



family interval / image interval width: min 6.9x  median 11.6x  max 26.1x

4 comparison(s) significant on images but NOT across families:
  C4 linear / raw q vs C5: per-family deltas [-0.458, -0.122, -0.073, -0.074, 0.434, -0.155]
  C7q bounded / raw q vs C2: per-family deltas [-0.057, -0.103, -0.093, -0.121, 0.272, 0.004]
  C7q bounded / raw q vs C5: per-family deltas [-0.457, -0.112, -0.075, -0.077, 0.28, -0.159]
  C8b bounded / class + q vs C5: per-family deltas [-0.09, -0.137, -0.065, -0.07, 0.197, -0.084]
  These are claims about these six corruptions, not about corruption in general.


In [ ]:
print("="*100)
MATRIX=[]
for A in SOUND_Q:
    r2=T[(T.arm==A)&(T.vs==REF[0])].iloc[0]; r5=T[(T.arm==A)&(T.vs==REF[1])].iloc[0]
    MATRIX.append(dict(arm=A,selected=A in WINNERS,
        nll_C2_img=bool(r2.win_nll_image), nll_C2_fam=bool(r2.win_nll_family),
        ece_C2_fam=bool(r2.win_ece_family),
        nll_C5_img=bool(r5.win_nll_image), nll_C5_fam=bool(r5.win_nll_family),
        ece_C5_fam=bool(r5.win_ece_family),
        score_image=int(r2.win_nll_image)+int(r5.win_nll_image),
        score_family=int(r2.win_nll_family)+int(r2.win_ece_family)
                    +int(r5.win_nll_family)+int(r5.win_ece_family)))
M=pd.DataFrame(MATRIX).sort_values(["score_family","score_image"],ascending=False)
display(M); M.to_csv(OUT/"phase0_11_matrix.csv",index=False)

SEL_VALID = (RHO_BEST >= 0.6)
print("="*100)
if not SEL_VALID:
    verdict="UNDETERMINED — the selection rule is invalid"
    guidance=(f"No selection rule ranks arms consistently with the test outcome (best Spearman "
        f"{RHO_BEST:+.3f} against a 0.60 threshold), so no arm can be named 'the method' from "
        f"this evidence. Reading the verdict off whatever the rule happened to pick is how "
        f"Phase 0.6, 0.7, 0.9 and 0.10 each produced a wrong conclusion.\n\n"
        f"The cause is not a patchable bug: corruption families are not exchangeable, so a rule "
        f"tuned on some families cannot rank arms for others. Fix the DESIGN before the full "
        f"experiment -- either enlarge the family pool substantially so held-out families are "
        f"representative, or select on a criterion that does not require family transfer at all "
        f"(for example the simplest arm that is sound in every fold).")
elif M.iloc[0]["score_family"]>=4:
    verdict="METHOD CONTRIBUTION SUPPORTED UNDER FAMILY-LEVEL UNCERTAINTY"
    guidance=(f"{M.iloc[0]['arm']} beats both baselines on both metrics with intervals that "
        f"account for family-to-family variance, not just image variance. That is the strong "
        f"form of the claim and it is what a reviewer will test.")
elif M.iloc[0]["score_family"]>=1:
    verdict="SUPPORTED ON THESE SIX CORRUPTIONS ONLY"
    guidance=(f"{M.iloc[0]['arm']} wins on the image-level intervals but the family-level "
        f"intervals do not all exclude zero. State the scope precisely: the result holds on the "
        f"six corruption families tested, and the evidence does not support generalisation to "
        f"corruption in general. Report the per-family deltas so the reader sees the spread.")
else:
    verdict="NO METHOD CONTRIBUTION UNDER FAMILY-LEVEL UNCERTAINTY"
    guidance=("Once family-to-family variance is accounted for, no arm beats a scalar "
        "temperature fitted on augmented validation data. Reframe around SDI-C, the split "
        "protocol, the leakage precondition and the corrected severity ladders. Report the "
        "negative result explicitly -- it is more useful than an overstated positive one.")
failed=list(agg[~agg.gate.isin(["sound","reference"])].index)
print(f"\nVERDICT: {verdict}\n\n{guidance}")
if failed: print(f"\nExcluded (failed the gate): {failed}")
json.dump({"verdict":verdict,"selection_valid":bool(SEL_VALID),"rho_best":float(RHO_BEST),
           "winners":WINNERS,"guidance":guidance,"excluded":failed,
           "matrix":M.to_dict("records"),"comparisons":T.to_dict("records")},
          open(OUT/"phase0_11_verdict.json","w"),indent=2)

,arm,selected,nll_C2_img,nll_C2_fam,ece_C2_fam,nll_C5_img,nll_C5_fam,ece_C5_fam,score_image,score_family
3,C7q bounded / raw q,False,True,False,False,True,False,False,2,0
0,C4 linear / raw q,False,False,False,False,True,False,False,1,0
5,C8b bounded / class + q,True,False,False,False,True,False,False,1,0
1,C6c linear / std (robust),False,False,False,False,False,False,False,0,0
2,C7 bounded / std (robust),False,False,False,False,False,False,False,0,0
4,C8 linear / class + q,True,False,False,False,False,False,False,0,0



VERDICT: UNDETERMINED — the selection rule is invalid

No selection rule ranks arms consistently with the test outcome (best Spearman +0.257 against a 0.60 threshold), so no arm can be named 'the method' from this evidence. Reading the verdict off whatever the rule happened to pick is how Phase 0.6, 0.7, 0.9 and 0.10 each produced a wrong conclusion.

The cause is not a patchable bug: corruption families are not exchangeable, so a rule tuned on some families cannot rank arms for others. Fix the DESIGN before the full experiment -- either enlarge the family pool substantially so held-out families are representative, or select on a criterion that does not require family transfer at all (for example the simplest arm that is sound in every fold).

Excluded (failed the gate): ['C3 MLP / raw q']


---

## What to carry into the paper

Whatever the verdict, four results are now established on five folds with image-level intervals
and no test-set peeking, and all four belong in the manuscript.

**Clean-fit calibration is worse than no calibration.** Fitted on clean validation data, scalar
temperature scaling drives *T* below 1 and *sharpens* an already-overconfident model. Report the
number; it motivates the entire paper in one line.

**The MLP temperature head fails.** 833 parameters, temperature pinned to the floor, likelihood
worse than uncalibrated. It is ablation evidence against capacity, not a component.

**C5 is the load-bearing control.** Any claim that a temperature tracks image quality rather than
class identity has to survive a calibrator that is handed the predicted class and nothing else.

**The leakage-corrected arm trades likelihood for calibration error.** Report both metrics for
every arm rather than the one that flatters the method.

## Still blocked, and not affected by any of this

KSDD2 from the official ViCoS release. Magnetic Tile folds rebuilt without the discarded pHash
grouping. The F1–F6 fixes applied to the main notebook. And this remains one backbone on one
dataset — the full experiment must show the result replicates across backbones and datasets
before any of it is a paper claim.